In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np
# from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Arial"
import seaborn as sns
# from joblib import Parallel, delayed
import re
import os
from itertools import combinations
# from statsmodels.stats.multitest import multipletests


The purpose of this notebook is to create

- Figure 2a: a bar plot comparing the performance (precision and recall) of the models on the cogntive status tasks (NC/MCI/DE), with stats using bootstrap.
- A table containing the same data, plus the F1 score. The table is in LaTeX format, ready to be pasted into the manuscript.

Notice that we are only loading the COG task

In [2]:
# Define mappings
model_map = {
    "Qwen2.5-3B-Instruct": "Q3B",
    "NACC-3B": "LUNAR-OS-SCe",
    "NACC-3B-SCE": "LUNAR-OS",
    "NACC-3B-OS": "LUNAR-SCe",
    "NACC-3B-OS-SCE": "LUNAR",
    "Qwen2.5-7B-Instruct": "Q7B",
}

class_map = {
    "Not applicable (no cognitive impairment)": "NC",
    "Alzheimer's disease (AD)": "AD",
    "Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)": "FTLD",
    "Lewy body disease (LBD)": "LBD",
    "Vascular brain injury or vascular dementia including stroke (VD)": "VD",
    "Idiopathic Parkinson's Disease": "Idiopathic PD",
    "Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)": "SEF",
    "Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)": "PSY",
    "Other (Multiple system atrophy, Essential tremor, Down syndrome, Huntington's disease, Prion disease, Traumatic brain injury, Normal-pressure hydrocephalus, Epilepsy, CNS neoplasm, etc)": "Other",
    "Prodromal Parkinson's Disease": "Prodromal PD",
    "No PD nor other neurological disorder": "No PD/ND",
    "Other neurological disorder(s)": "Other ND"
}

# Load the data

In [3]:
nifd_path = Path('/projectnb/vkolagrp/projects/adrd_foundation_model/results/NIFD/test_etpr')
nacc_path = Path('/projectnb/vkolagrp/projects/adrd_foundation_model/results/NACC/test_etpr')
adni_path = Path('/projectnb/vkolagrp/projects/adrd_foundation_model/results/ADNI/test_etpr')
ppmi_path = Path('/projectnb/vkolagrp/projects/adrd_foundation_model/results/PPMI/test_etpr')
brainlat_path = Path('/projectnb/vkolagrp/projects/adrd_foundation_model/results/brainlat/test_etpr')

In [5]:
def option_string_to_dict(options):
    # The option string is randomized (e.g. NC is not always option A). We need to break down the 
    # options and look at the text (e.g. MCI), not just the letter that identifies them in a particular question
    pattern = r"([A-Z])\. ([^\n]+)"
    matches = re.findall(pattern, options)
    return {key: value for key, value in matches}

def load_answers(dir_path, dataset_name):
    # load all parquet files from the directory, stack them into a pandas datafame
    # this only reads the participant ID, ground trush answer and the prediction. 
    # Reading only those columns is significantly faster (about 100x) than loading the whole dataframe.
    # Loading everything is very slow because there are extremely long strings (model outputs) in some columns

    fpaths = list(dir_path.rglob('*.parquet'))

    dfs = []

    cols_to_read = ['ID','ground_truth','prediction','ground_truth_text','options']#, 'generated_text']

    for fpath in tqdm(fpaths):

        model = fpath.parent.name.split('-',3)[-1] 
        benchmark = fpath.parent.parent.name.split('_',1)[-1].upper()

        df = pd.read_parquet(fpath,columns=cols_to_read)
    
        df = df.assign(model=model, benchmark=benchmark)

        df['correct'] = (df['ground_truth'] == df['prediction']).astype(int)

        df['prediction_text'] = df.apply(lambda row: option_string_to_dict(row['options']).get(row['prediction'],'invalid'),axis=1)

        dfs.append(df)

    df = pd.concat(dfs)
    df['dataset'] = dataset_name

    # make these columns Categorical
    group_cols = ["dataset", "benchmark", "model", "ground_truth_text", 'prediction_text']
    for col in group_cols:
        df[col] = pd.Categorical(df[col])

    # keep only the models we actually care about
    df = df[df['model'].isin(list(model_map.keys()))]

    return df


The `dataset_name` parameter will be used in the results dataset to identify which dataset the data came from

In [6]:
nacc_res = load_answers(nacc_path,dataset_name='NACC')

  0%|          | 0/8 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:11<00:00,  1.46s/it]


In [7]:
nacc_res.prediction_text.unique()

['Alzheimer's disease (AD)', 'Not applicable (no cognitive impairment)', 'Vascular brain injury or vascular dementia in..., 'Psychiatric conditions including schizophreni..., 'Lewy body disease (LBD)', 'Systemic and environmental factors including ..., 'Other (Multiple system atrophy, Essential tre..., 'Frontotemporal lobar degeneration and its var..., 'invalid']
Categories (9, object): ['Alzheimer's disease (AD)', 'Frontotemporal lobar degeneration and its var..., 'Lewy body disease (LBD)', 'Not applicable (no cognitive impairment)', ..., 'Psychiatric conditions including schizophreni..., 'Systemic and environmental factors including ..., 'Vascular brain injury or vascular dementia in..., 'invalid']

In [8]:
brainlat_res = load_answers(brainlat_path,dataset_name='BrainLat')

100%|██████████| 8/8 [00:00<00:00, 26.05it/s]


In [9]:
ppmi_res = load_answers(ppmi_path,dataset_name='PPMI')

100%|██████████| 8/8 [00:00<00:00, 34.12it/s]


In [10]:
nifd_res = load_answers(nifd_path,dataset_name='NIFD')

100%|██████████| 8/8 [00:00<00:00, 49.71it/s]


In [11]:
adni_res = load_answers(adni_path,dataset_name='ADNI')

100%|██████████| 8/8 [00:00<00:00, 17.27it/s]


In [12]:
# concatenate everything in a tall format dataframe

results_df = pd.concat(
    [
        nacc_res,
        adni_res,
        brainlat_res,
        nifd_res,
        ppmi_res
    ]
).reset_index(drop=True)

In [13]:
results_df['dataset_raw'] = results_df['dataset']

# results_df['dataset'] = results_df['dataset'].replace(
#     {
#         'ADNI':'Other',
#         'BrainLat':'Other',
#         'NIFD':'Other',
#         'PPMI':'Other',
#     }
# )
results_df["trial"] = results_df.index

In [14]:
results_df.head()

,ID,ground_truth,prediction,ground_truth_text,options,model,benchmark,correct,prediction_text,dataset,dataset_raw,trial
0,NACC344354,G,G,Alzheimer's disease (AD),A. Not applicable (no cognitive impairment)\nB...,Qwen2.5-7B-Instruct,ETPR,1,Alzheimer's disease (AD),NACC,NACC,0
1,NACC344354,G,A,Alzheimer's disease (AD),A. Not applicable (no cognitive impairment)\nB...,Qwen2.5-7B-Instruct,ETPR,0,Not applicable (no cognitive impairment),NACC,NACC,1
2,NACC344354,G,D,Alzheimer's disease (AD),A. Not applicable (no cognitive impairment)\nB...,Qwen2.5-7B-Instruct,ETPR,0,Vascular brain injury or vascular dementia inc...,NACC,NACC,2
3,NACC344354,G,A,Alzheimer's disease (AD),A. Not applicable (no cognitive impairment)\nB...,Qwen2.5-7B-Instruct,ETPR,0,Not applicable (no cognitive impairment),NACC,NACC,3
4,NACC344354,G,G,Alzheimer's disease (AD),A. Not applicable (no cognitive impairment)\nB...,Qwen2.5-7B-Instruct,ETPR,1,Alzheimer's disease (AD),NACC,NACC,4


In [15]:
results_df['model'].unique()

['Qwen2.5-7B-Instruct', 'NACC-3B-OS-SCE', 'NACC-3B-SCE', 'NACC-3B', 'NACC-3B-OS', 'Qwen2.5-3B-Instruct']
Categories (8, object): ['NACC-3B', 'NACC-3B-OS', 'NACC-3B-OS-SCE', 'NACC-3B-SCE', 'NACC-7B-OS', 'NACC-7B-OS-SCE', 'Qwen2.5-3B-Instruct', 'Qwen2.5-7B-Instruct']

# Metrics

In [16]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from itertools import combinations
from statsmodels.stats.multitest import multipletests

def _vectorized_metric_calc(y_true, y_pred, label_code, metric):
    """Core vectorized math for both bootstrap and permutation."""
    tp = np.sum((y_pred == label_code) & (y_true == label_code), axis=-1)
    
    if metric == 'precision':
        den = np.sum(y_pred == label_code, axis=-1)
    elif metric == 'recall':
        den = np.sum(y_true == label_code, axis=-1)
    elif metric == 'f1':
        precision_den = np.sum(y_pred == label_code, axis=-1)
        recall_den = np.sum(y_true == label_code, axis=-1)
        precision = np.divide(
            tp, precision_den, out=np.zeros_like(tp, dtype=float), where=precision_den != 0
        )
        recall = np.divide(
            tp, recall_den, out=np.zeros_like(tp, dtype=float), where=recall_den != 0
        )
        with np.errstate(divide='ignore', invalid='ignore'):
            f1 = np.divide(
                2 * precision * recall,
                precision + recall,
                out=np.zeros_like(tp, dtype=float),
                where=(precision + recall) != 0,
            )
        return f1
    else:
        raise ValueError(f"Unknown metric: {metric}")
    return np.divide(tp, den, out=np.zeros_like(tp, dtype=float), where=den != 0)



# Bootstrap

In [17]:
def _single_bootstrap_task(group_info, y_true_b, y_pred_b, lbl_code, m_type):
    """Worker for individual bootstrap metric tasks to saturate 20+ cores."""
    boot_values = _vectorized_metric_calc(y_true_b, y_pred_b, lbl_code, m_type)
    low, med, high = np.percentile(boot_values, [2.5, 50, 97.5])
    return {
        **group_info, "class_code": lbl_code, "metric": m_type,
        "mean": np.mean(boot_values), "median": med, "low": low, "high": high
    }

def optimized_bootstrap_parallel(df, n_boot=1000, seed=42, n_jobs=-1):
    """
    Bootstrap CIs for precision/recall per (dataset, benchmark, model, class).
    """
    # Keep original text columns for grouping
    df_grouped = df[["ID", "trial", "dataset", "benchmark", "model", "ground_truth_text", "prediction_text"]].copy()
    
    # Build tasks
    groups = list(df_grouped.groupby(["dataset", "benchmark", "model"], observed=True))
    main_rng = np.random.default_rng(seed)
    all_tasks = []
    
    # We'll build a global int_to_label mapping after collecting all groups
    all_int_to_label = {}
    
    print(f"Preparing Bootstrap data for {len(groups)} groups...")
    
    for i, (g_id, group) in enumerate(groups):
        # Sort by ID for consistency
        group = group.sort_values(['ID', "trial"]).reset_index(drop=True)
        
        # Get all unique categories for THIS group
        gt_cats = group["ground_truth_text"].astype("category").cat.categories.tolist()
        pred_cats = group["prediction_text"].astype("category").cat.categories.tolist()
        all_cats = sorted(set(gt_cats + pred_cats))
        
        # Ensure 'invalid' is included
        if "invalid" not in all_cats:
            all_cats.append("invalid")
        
        # Create shared categorical dtype for this group
        cat_dtype = pd.CategoricalDtype(categories=all_cats, ordered=False)
        invalid_code = all_cats.index("invalid")
        
        # Encode with consistent mapping
        y_true = group["ground_truth_text"].astype(cat_dtype).cat.codes.astype(np.int16).to_numpy()
        y_pred = group["prediction_text"].astype(cat_dtype).cat.codes.astype(np.int16).to_numpy()
        
        # Labels to process (exclude invalid)
        labels_to_process = [i for i in range(len(all_cats)) if i != invalid_code]
        
        # Store mapping for this group (we'll merge later)
        group_key = (g_id[0], g_id[1], g_id[2])
        all_int_to_label[group_key] = {i: cat for i, cat in enumerate(all_cats)}
        
        group_seed = int(main_rng.integers(0, 2**32))
        rng = np.random.default_rng(group_seed)
        
        # Generate indices once per group
        indices = rng.integers(0, len(y_true), size=(n_boot, len(y_true)))
        y_true_b = y_true[indices]
        y_pred_b = y_pred[indices]
        
        for lbl_code in labels_to_process:
            for m_type in ['precision', 'recall', 'f1']:
                all_tasks.append((
                    {"dataset": g_id[0], "benchmark": g_id[1], "model": g_id[2]},
                    y_true_b, y_pred_b, lbl_code, m_type
                ))
    
    print(f"Executing Bootstrap on {len(all_tasks)} tasks across {n_jobs} cores...")
    results = Parallel(n_jobs=n_jobs)(delayed(_single_bootstrap_task)(*t) for t in all_tasks)
    
    res_df = pd.DataFrame(results)
    
    # Map class codes back to labels using group-specific mappings
    res_df['class'] = res_df.apply(
        lambda row: all_int_to_label[(row['dataset'], row['benchmark'], row['model'])][row['class_code']],
        axis=1
    )
    
    return res_df.drop(columns=['class_code'])

In [18]:
# Step 1: Compute bootstrap CIs with samples
all_metrics = optimized_bootstrap_parallel(
    results_df, n_boot=1000, seed=42, n_jobs=20
)

Preparing Bootstrap data for 30 groups...
Executing Bootstrap on 648 tasks across 20 cores...


In [19]:
all_metrics.sort_values(['dataset','model','metric','class'])[:20]

,dataset,benchmark,model,metric,mean,median,low,high,class
2,ADNI,ETPR,NACC-3B,f1,0.760841,0.760761,0.747561,0.773784,Alzheimer's disease (AD)
5,ADNI,ETPR,NACC-3B,f1,0.000000,0.000000,0.000000,0.000000,Frontotemporal lobar degeneration and its vari...
8,ADNI,ETPR,NACC-3B,f1,0.000000,0.000000,0.000000,0.000000,Lewy body disease (LBD)
11,ADNI,ETPR,NACC-3B,f1,0.784054,0.784295,0.771888,0.795911,Not applicable (no cognitive impairment)
14,ADNI,ETPR,NACC-3B,f1,0.000000,0.000000,0.000000,0.000000,"Other (Multiple system atrophy, Essential trem..."
17,ADNI,ETPR,NACC-3B,f1,0.000000,0.000000,0.000000,0.000000,Psychiatric conditions including schizophrenia...
20,ADNI,ETPR,NACC-3B,f1,0.000000,0.000000,0.000000,0.000000,Systemic and environmental factors including i...
23,ADNI,ETPR,NACC-3B,f1,0.000000,0.000000,0.000000,0.000000,Vascular brain injury or vascular dementia inc...
0,ADNI,ETPR,NACC-3B,precision,0.774096,0.774014,0.757561,0.790218,Alzheimer's disease (AD)
3,ADNI,ETPR,NACC-3B,precision,0.000000,0.000000,0.000000,0.000000,Frontotemporal lobar degeneration and its vari...


In [20]:
all_metrics = all_metrics[all_metrics['mean'] != 0].reset_index(drop=True)

# Latex

In [21]:
def generate_latex_table(all_metric, model_map, class_map):
    df = all_metric.copy()
    
    # 1. Apply Mappings
    df['model'] = df['model'].map(model_map).fillna(df['model'])
    df['class'] = df['class'].map(class_map).fillna(df['class'])

    # 2. Pivot to get metrics as columns for F1 calculation
    stats = df.pivot_table(
        index=['dataset', 'class', 'model'],
        columns='metric',
        values=['median', 'low', 'high']
    ).reset_index()

    # Flatten multi-index columns: e.g., ('median', 'precision') -> 'precision_median'
    stats.columns = [f"{col[1]}_{col[0]}" if col[1] else col[0] for col in stats.columns]

    # 4. Define Categorical Sort Order
    model_order = list(model_map.values())
    
    stats['class'] = pd.Categorical(stats['class'], categories=class_map.values(), ordered=True)
    stats['model'] = pd.Categorical(stats['model'], categories=model_order, ordered=True)
    
    # Sort and reset index so loop index 'i' matches the dataframe index
    stats = stats.sort_values(by=['dataset', 'class', 'model']).reset_index(drop=True)

    # 5. Identify the best performing model (highest median) for each metric per group
    metrics = ['precision', 'recall', 'f1']
    best_lookup = set()
    
    # for m in metrics:
    #     # Pass observed=False to satisfy the future pandas requirement
    #     idx = stats.groupby(['dataset', 'class'], observed=True)[f'{m}_median'].idxmax()
    #     for i in idx:
    #         # Check for NaN to handle groups that might be empty after categorical mapping
    #         if pd.notna(i):
    #             best_lookup.add((i, m))
                
    best_lookup = set()
    second_best_lookup = set()

    for m in metrics:
        col = f"{m}_median"

        # Rank within each (dataset, class), highest value = rank 1
        ranks = stats.groupby(
            ["dataset", "class"], observed=False
        )[col].rank(method="first", ascending=False)

        # Best (rank == 1)
        best_idx = stats.index[(ranks == 1) & stats[col].notna()]
        for i in best_idx:
            best_lookup.add((i, m))

        # Second best (rank == 2)
        second_idx = stats.index[(ranks == 2) & stats[col].notna()]
        for i in second_idx:
            second_best_lookup.add((i, m))

    # 6. Build LaTeX manually
    headers = ['Dataset', 'Class', 'Model', 'Precision', 'Recall', 'F1-score']
    latex_lines = [
        "\\begin{table}[ht]",
        "\\centering",
        "\\small",
        "\\begin{tabular}{lllccc}",
        "\\hline",
        " & ".join(headers) + " \\\\",
        "\\hline"
    ]

    prev_ds, prev_cl = None, None

    for i, row in stats.iterrows():
        # Visual grouping: Add midrule when Class changes
        if prev_cl is not None and row['class'] != prev_cl:
            latex_lines.append("\\hline")
            
        ds_disp = row['dataset'] if row['dataset'] != prev_ds else ""
        cl_disp = row['class'] if row['class'] != prev_cl else ""
        
        # Format each metric column
        formatted_metrics = []
        for m in metrics:
            val_str = f"{row[f'{m}_median']:.3f} [{row[f'{m}_low']:.3f}, {row[f'{m}_high']:.3f}]"
            if (i, m) in best_lookup:
                val_str = f"\\textbf{{{val_str}}}"
            if (i, m) in second_best_lookup:
                val_str = f"\\underline{{{val_str}}}"
            formatted_metrics.append(val_str)
        
        row_str = f"{ds_disp} & {cl_disp} & {row['model']} & " + " & ".join(formatted_metrics) + " \\\\"
        latex_lines.append(row_str)
        
        prev_ds, prev_cl = row['dataset'], row['class']

    latex_lines.extend(["\\hline", "\\end{tabular}", "\\end{table}"])
    return "\n".join(latex_lines)

In [22]:
latex_output = generate_latex_table(all_metrics, model_map=model_map, class_map=class_map)
print(latex_output)

\begin{table}[ht]
\centering
\small
\begin{tabular}{lllccc}
\hline
Dataset & Class & Model & Precision & Recall & F1-score \\
\hline
ADNI & NC & Q3B & 0.864 [0.840, 0.885] & 0.273 [0.257, 0.290] & 0.415 [0.395, 0.435] \\
 &  & LUNAR-OS-SCe & 0.847 [0.832, 0.862] & 0.730 [0.714, 0.745] & 0.784 [0.772, 0.796] \\
 &  & LUNAR-OS & 0.855 [0.841, 0.869] & \underline{0.736 [0.720, 0.752]} & \underline{0.791 [0.778, 0.804]} \\
 &  & LUNAR-SCe & 0.860 [0.847, 0.874] & \textbf{0.740 [0.723, 0.756]} & \textbf{0.795 [0.783, 0.807]} \\
 &  & LUNAR & \underline{0.865 [0.851, 0.878]} & 0.722 [0.706, 0.740] & 0.787 [0.774, 0.799] \\
 &  & Q7B & \textbf{0.887 [0.870, 0.902]} & 0.538 [0.520, 0.556] & 0.669 [0.653, 0.685] \\
\hline
 & AD & Q3B & 0.599 [0.581, 0.616] & 0.742 [0.724, 0.760] & 0.663 [0.648, 0.677] \\
 &  & LUNAR-OS-SCe & 0.774 [0.758, 0.790] & 0.748 [0.730, 0.764] & \underline{0.761 [0.748, 0.774]} \\
 &  & LUNAR-OS & 0.776 [0.760, 0.793] & \underline{0.775 [0.760, 0.791]} & \textbf{0.776 [